### lora fine-tuning notebook
this notebook provides a generalized pipeline to fine-tune any causal language model (such as qwen or llama) using parameter-efficient fine-tuning (lora/qlora) with the baseten-recommended post-training hyperparameters.

In [ ]:
# install required packages
!pip install -U torch transformers peft datasets accelerate bitsandbytes trl huggingface_hub

In [ ]:
# log in to hugging face hub to pull private models/push adapters
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig

# --- configuration setup ---
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # target hf base model
DATASET_ID = "yahma/alpaca-cleaned"      # dataset path or local json file
OUTPUT_DIR = "/data/axeai_lora_output"     # local save path
HUB_REPO_ID = None                       # e.g., "your_username/adapter_name"

# baseten-recommended hyperparameter recipe
EPOCHS = 2
LEARNING_RATE = 1e-3
LORA_RANK = 64
LORA_ALPHA = 32
GRADIENT_ACCUMULATION_STEPS = 16
USE_QLORA = True                         # use 4-bit quantization to fit in vram
MAX_LENGTH = 512

In [ ]:
# load model and tokenizer
print(f"loading tokenizer and base model: {MODEL_ID}")
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = "<|endoftext|>"

model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
    "dtype": compute_dtype,
}

if USE_QLORA:
    print("enabling qlora...")
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

In [ ]:
# load and format dataset
print(f"loading dataset: {DATASET_ID}")
if DATASET_ID.endswith(".json") or os.path.exists(DATASET_ID):
    dataset = load_dataset("json", data_files=DATASET_ID, split="train")
else:
    dataset = load_dataset(DATASET_ID, split="train")

# convert instructions to chat template conversation if necessary
sample_entry = dataset[0]
if "messages" not in sample_entry and "instruction" in sample_entry and "output" in sample_entry:
    print("formatting dataset to chat ml format...")
    def format_to_chat(example):
        user_content = example["instruction"]
        if example.get("input") and example["input"].strip() != "" and example["input"].strip() != "<noinput>":
            user_content += f"\n\nInput:\n{example['input']}"
        return {
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": example["output"]}
            ]
        }
    dataset = dataset.map(format_to_chat, remove_columns=dataset.column_names)

In [ ]:
# set up training adapters
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy="no",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_length=MAX_LENGTH,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)

In [ ]:
# start training
print("starting training loop...")
trainer.train()

# save model locally
print(f"saving fine-tuned adapter locally to: {OUTPUT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
print("saved locally!")

In [ ]:
# optional push to hugging face hub
if HUB_REPO_ID:
    print(f"pushing to hugging face hub repository: {HUB_REPO_ID}")
    try:
        trainer.push_to_hub(HUB_REPO_ID)
        print("successfully pushed model to hub!")
    except Exception as e:
        print(f"failed to push: {e}")